In [3]:
import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"
DERIVADO_PRO183 = "DERIVADO_PRO183"
RECHAZADO_RECEPCION = "RECHAZADO_RECEPCION"

# -------------------------
# PRO200 – Recepción de Materiales a Distancia (1° versión – Feb-2024)
# Motor HMI/Checklist operacional on-site + validaciones a distancia.
#
# Etapas del PRO200:
#   1) Revisión
#   2) Inspección
#   3) Almacenamiento
#   4) Rechazo / Devolución (PRO183)
#
# Convención:
# - type = "stage"        : encabezado de etapa + botón para iniciar
# - type = "task"         : Acción a ejecutar + Validación
# - type = "decision"     : rombo (XOR): pregunta obligatoria
# - type = "parallel_gate": rombo (+): ramas paralelas obligatorias + sincronización (AND)
# - type = "stage_end"    : cierre de etapa + botón para continuar a siguiente etapa
# - type = "end"          : fin/derivación total del proceso
# -------------------------

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

# -------------------------
# NODOS (flujos por etapa)
# -------------------------

NODOS = {
    # =============================
    # ETAPA 1 — REVISIÓN
    # =============================
    "S1_START": {
        "type": "stage",
        "etapa_num": 1,
        "etapa_nombre": "REVISIÓN",
        "mensaje": "Esta etapa abarca desde la recepción física del material en terreno, revisión de material y documentación, paralelos de descarga y revisión remota, hasta registrar la recepción en stock bloqueado y determinar si requiere inspección técnica.",
        "next": "R1_verificar_material_y_doc",
    },

    "R1_verificar_material_y_doc": {
        "type": "task",
        "titulo": "Verificar material y documentación",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Recepción física en terreno. Revisar que el material y su documentación (guía/factura/certificados/HDS si aplica) correspondan a lo solicitado.",
        "acciones": [
            "Verificar físicamente el material recibido (identificación, cantidades, integridad).",
            "Verificar documentación asociada disponible: guía de despacho / factura / certificados / HDS (si aplica según material).",
        ],
        "validacion": "¿Material y documentación fueron verificados en terreno (identidad, cantidad, integridad y documentos disponibles)?",
        "next": "D1_es_lo_solicitado",
    },

    "D1_es_lo_solicitado": {
        "type": "decision",
        "titulo": "¿Es lo solicitado?",
        "rol": "Supervisor O&M Solar (on-site)",
        "pregunta": "¿El material y su documentación corresponden a lo solicitado (OC/SOLPED) y están en condición de recepcionar?",
        "opciones": [
            {"label": "SÍ → Material recibido", "next": "G1_post_material_recibido"},
            {"label": "NO → Material rechazado (saltar a Etapa 4 Rechazo)", "next": "S4_START"},
        ],
        "ayuda": "Si NO corresponde a lo solicitado, el flujo indica Rechazo y derivación a devolución (PRO183).",
    },

    # Gate paralelo 1 (rombo +): documentación/aviso vs descarga
    "G1_post_material_recibido": {
        "type": "parallel_gate",
        "titulo": "Punto paralelo: Documentación + Descarga",
        "rol": "Supervisor O&M Solar (on-site) / Especialista de Almacén (a distancia)",
        "descripcion": "Desde 'Material recibido' se ejecutan actividades en paralelo. Debes completar TODAS las ramas para continuar.",
        "ramas": [
            {
                "label": "Rama A (corta): Documentación + Notificación",
                "start": "R2_digitalizar_archivar_doc",
                "flags_done": ["S1_DOC_ARCHIVADA", "S1_NOTIFICADO_RECEPCION"],
            },
            {
                "label": "Rama B (más larga): Descarga + (si aplica) apoyo remoto",
                "start": "R3_coordinar_descarga",
                "flags_done": ["S1_MATERIAL_DESCARGADO"],
            },
        ],
        "next": "R4_determinar_necesidad_inspeccion",
    },

    "R2_digitalizar_archivar_doc": {
        "type": "task",
        "titulo": "Digitalizar y archivar documentación",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Digitalizar y archivar documentación de recepción (HDS, certificados, factura/guía) según corresponda.",
        "acciones": [
            "Digitalizar documentos disponibles (foto/scan legible).",
            "Archivar/respaldar documentación según estándar vigente del sitio (repositorio/carpeta definida).",
        ],
        "validacion": "¿La documentación fue digitalizada y archivada/respaldada de forma legible?",
        "set_flags": {"S1_DOC_ARCHIVADA": True},
        "next": "R2b_notificar_recepcion_material",
    },

    "R2b_notificar_recepcion_material": {
        "type": "task",
        "titulo": "Notificar recepción de material",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Notificar la recepción del material para activar revisión remota y continuidad del flujo.",
        "acciones": [
            "Enviar notificación de recepción de material según canal definido (correo/medio interno).",
            "Incluir identificación de OC, proveedor, guía/factura y respaldo digital disponible.",
        ],
        "validacion": "¿La recepción fue notificada con identificación de OC/proveedor y respaldo digital adjunto o referenciado?",
        "set_flags": {"S1_NOTIFICADO_RECEPCION": True},
        "next": "G1_RETURN",
    },

    "R3_coordinar_descarga": {
        "type": "task",
        "titulo": "Coordinar descarga de material",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Coordinar la descarga segura del material. Si requiere apoyo (carga pesada/equipos), activar apoyo remoto/planificado.",
        "acciones": [
            "Definir plan de descarga seguro (personas/equipos/área).",
            "Verificar condiciones de descarga/manipulación (peso, izaje, sustancias peligrosas si aplica).",
            "Si requiere recursos excepcionales (grúa/horquilla): coordinar apoyo según disponibilidad.",
        ],
        "validacion": "¿La descarga fue coordinada con control de riesgos y recursos necesarios definidos?",
        "next": "D2_requiere_apoyo_descarga",
    },

    "D2_requiere_apoyo_descarga": {
        "type": "decision",
        "titulo": "¿Requiere apoyo?",
        "rol": "Supervisor O&M Solar (on-site)",
        "pregunta": "¿La descarga requiere apoyo adicional (p.ej., equipos/recursos excepcionales)?",
        "opciones": [
            {"label": "NO → Continuar con descarga", "next": "R3b_confirmar_material_descargado"},
            {"label": "SÍ → Solicitar/apoyar descarga (rama con apoyo remoto)", "next": "R3a_apoyar_descarga_remoto"},
        ],
        "ayuda": "El diagrama contempla apoyo del Especialista de Almacén (a distancia) en coordinación de descarga cuando aplica.",
    },

    "R3a_apoyar_descarga_remoto": {
        "type": "task",
        "titulo": "Apoyar descarga de material",
        "rol": "Especialista de Almacén (a distancia) / Supervisor O&M Solar",
        "descripcion": "Coordinar apoyo para descarga (p.ej., equipos, priorización, recursos).",
        "acciones": [
            "Confirmar si la OC requiere recursos excepcionales para recepción.",
            "Coordinar disponibilidad de equipos/recursos para descarga (p.ej., grúa/horquilla) con el sitio.",
            "Acordar ventana/condiciones para la descarga segura.",
        ],
        "validacion": "¿El apoyo para descarga quedó coordinado (equipos/recursos confirmados y ventana definida)?",
        "next": "R3b_confirmar_material_descargado",
    },

    "R3b_confirmar_material_descargado": {
        "type": "task",
        "titulo": "Confirmar material descargado",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Confirmar que el material fue descargado y quedó en condición segura para continuar el proceso.",
        "acciones": [
            "Ejecutar descarga según plan.",
            "Verificar que el material quedó descargado en área segura y controlada.",
        ],
        "validacion": "¿El material quedó descargado en condición segura y controlada?",
        "set_flags": {"S1_MATERIAL_DESCARGADO": True},
        "next": "G1_RETURN",
    },

    # Retorno lógico al gate (sin stack extra)
    "G1_RETURN": {
        "type": "parallel_gate",
        "titulo": "Volver a Punto paralelo (sincronización)",
        "rol": "",
        "descripcion": "Regresa al punto paralelo para completar la(s) rama(s) pendiente(s).",
        "ramas": [],
        "next": "G1_post_material_recibido",
        "hidden": True
    },

    "R4_determinar_necesidad_inspeccion": {
        "type": "task",
        "titulo": "Determinar necesidad de inspección técnica",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Determinar si el material requiere inspección técnica (según naturaleza del material / criterio técnico).",
        "acciones": [
            "Revisar el material recibido y definir si requiere inspección técnica.",
            "Registrar/consignar determinación para continuidad del flujo.",
        ],
        "validacion": "¿Se determinó (y registró) si el material requiere inspección técnica?",
        "next": "D3_requiere_inspeccion",
    },

    "D3_requiere_inspeccion": {
        "type": "decision",
        "titulo": "¿Material requiere inspección técnica?",
        "rol": "Supervisor O&M Solar (on-site)",
        "pregunta": "¿El material requiere inspección técnica?",
        "opciones": [
            {"label": "NO requiere inspección técnica", "next": "G2_post_notificacion_recepcion"},
            {"label": "SÍ requiere inspección técnica", "next": "G2_post_notificacion_recepcion"},
        ],
        "ayuda": "Esta determinación se usa en la Etapa 2 (Inspección).",
        "set_flags_by_choice": {
            "NO requiere inspección técnica": {"S2_REQ_INSPECCION": False},
            "SÍ requiere inspección técnica": {"S2_REQ_INSPECCION": True},
        }
    },

    # Gate paralelo 2 (rombo +) tras notificación de recepción:
    # Rama documentos -> registrar stock bloqueado
    # Rama maestro -> decisión enriquecer -> (si) enriquecer -> converge en registrar stock bloqueado
    "G2_post_notificacion_recepcion": {
        "type": "parallel_gate",
        "titulo": "Punto paralelo: Validaciones remotas (Documentos / Maestro)",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Tras la notificación de recepción, se realizan validaciones en paralelo. Ambas ramas deben quedar completadas para cerrar Etapa 1.",
        "ramas": [
            {
                "label": "Rama A: Verificar documentos → Registrar recepción en stock bloqueado",
                "start": "R5_verificar_documentos_recepcion",
                "flags_done": ["S1_DOCS_RECEPCION_OK", "S1_STOCK_BLOQUEADO_OK"],
            },
            {
                "label": "Rama B: Verificar maestro → ¿Requiere enriquecer? → (si) enriquecer",
                "start": "R6_verificar_maestro_materiales",
                "flags_done": ["S1_MAESTRO_OK", "S1_ENRIQUECIMIENTO_OK"],
            },
        ],
        "next": "S1_END",
    },

    "R5_verificar_documentos_recepcion": {
        "type": "task",
        "titulo": "Verificar documentos de recepción",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Verificar documentación de recepción (HDS, certificados, factura/guía) y consistencia con OC/material.",
        "acciones": [
            "Revisar HDS (si aplica), certificados y documento comercial (guía/factura) disponibles.",
            "Confirmar consistencia de documentos con OC/material recibido.",
        ],
        "validacion": "¿Los documentos de recepción fueron verificados y son consistentes con OC/material?",
        "set_flags": {"S1_DOCS_RECEPCION_OK": True},
        "next": "R5b_registrar_stock_bloqueado",
    },

    "R5b_registrar_stock_bloqueado": {
        "type": "task",
        "titulo": "Registrar recepción en stock bloqueado",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Registrar la recepción de material en stock bloqueado en SAP según corresponda a la etapa de revisión.",
        "acciones": [
            "Registrar recepción en stock bloqueado en SAP (según transacción definida en el PRO).",
            "Asegurar trazabilidad de OC/documentos asociados.",
        ],
        "validacion": "¿La recepción quedó registrada en stock bloqueado con trazabilidad (OC/documentos)?",
        "set_flags": {"S1_STOCK_BLOQUEADO_OK": True, "S1_ENLACE_STOCK_BLOQUEADO": True},
        "next": "G2_RETURN",
    },

    "R6_verificar_maestro_materiales": {
        "type": "task",
        "titulo": "Verificar maestro de materiales",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Verificar datos del maestro de materiales asociados al material recepcionado.",
        "acciones": [
            "Verificar datos de creación/maestro del material en SAP.",
            "Confirmar que características/atributos mínimos estén correctos para recepción y uso.",
        ],
        "validacion": "¿El maestro de materiales fue verificado?",
        "set_flags": {"S1_MAESTRO_OK": True},
        "next": "D4_requiere_enriquecer",
    },

    "D4_requiere_enriquecer": {
        "type": "decision",
        "titulo": "¿Requiere enriquecer material?",
        "rol": "Especialista de Almacén (a distancia)",
        "pregunta": "¿Se requiere enriquecer datos del material durante el proceso de recepción?",
        "opciones": [
            {"label": "NO → No se requiere enriquecer", "next": "R6c_cerrar_rama_maestro"},
            {"label": "SÍ → Enriquecer material (ZMM_MANT_CARACT)", "next": "R6b_enriquecer_material"},
        ],
        "ayuda": "El PRO contempla enriquecimiento de datos mediante transacción ZMM_MANT_CARACT cuando aplica.",
    },

    "R6b_enriquecer_material": {
        "type": "task",
        "titulo": "Enriquecer material",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Enriquecer los datos del material en SAP durante recepción.",
        "acciones": [
            "Ejecutar enriquecimiento de datos del material en SAP mediante transacción ZMM_MANT_CARACT.",
            "Verificar que los atributos enriquecidos queden guardados y trazables.",
        ],
        "validacion": "¿El material fue enriquecido en SAP (ZMM_MANT_CARACT) y quedó guardado correctamente?",
        "set_flags": {"S1_ENRIQUECIMIENTO_OK": True},
        "next": "G2_RETURN",
    },

    "R6c_cerrar_rama_maestro": {
        "type": "task",
        "titulo": "Cerrar rama maestro (sin enriquecimiento)",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Confirmar que no se requiere enriquecimiento adicional y cerrar la rama.",
        "acciones": [
            "Confirmar que el maestro verificado no requiere enriquecimiento adicional para continuar.",
        ],
        "validacion": "¿Se confirma que NO se requiere enriquecer material en esta recepción?",
        "set_flags": {"S1_ENRIQUECIMIENTO_OK": True},
        "next": "G2_RETURN",
    },

    "G2_RETURN": {
        "type": "parallel_gate",
        "titulo": "Volver a Punto paralelo (sincronización)",
        "rol": "",
        "descripcion": "Regresa al punto paralelo para completar la(s) rama(s) pendiente(s).",
        "ramas": [],
        "next": "G2_post_notificacion_recepcion",
        "hidden": True
    },

    "S1_END": {
        "type": "stage_end",
        "etapa_num": 1,
        "etapa_nombre": "REVISIÓN",
        "mensaje_fin": "Etapa 1 (Revisión) terminada: recepción registrada en stock bloqueado y determinación de inspección definida.",
        "next_stage": "S2_START",
    },

    # =============================

    # =============================
    # ETAPA 2 — INSPECCIÓN
    # (Basado en diagrama PRO200: determinación → notificación → registro/remoto → (si aplica) inspección → resultado → ubicación o devolución)
    # =============================
    "S2_START": {
        "type": "stage",
        "etapa_num": 2,
        "etapa_nombre": "INSPECCIÓN",
        "mensaje": "Etapa 2 (Inspección): se determina si el material requiere inspección técnica. Si requiere, se ejecuta inspección y se notifica resultado. En paralelo, Almacén (a distancia) registra/valida en sistema según flujo. Si NO aprueba, deriva a Etapa 4 (Rechazo/Devolución). Si aprueba o no requiere, pasa a Etapa 3 (Almacenamiento).",
        "next": "S2_G1_paralelo_inicio",
    },

    "S2_G1_paralelo_inicio": {
        "type": "parallel_gate",
        "titulo": "Punto paralelo ➕ (Inspección + Registro remoto)",
        "rol": "Supervisor O&M Solar (on-site) / Especialista de Almacén (a distancia)",
        "descripcion": "Este punto indica que hay dos líneas de trabajo que deben completarse: (A) decisión/inspección on-site y (B) registro/validación a distancia. El proceso avanza a Almacenamiento solo cuando ambas ramas estén completadas (o si se deriva a Rechazo).",
        "ramas": [
            {
                "label": "Rama A (on-site): Determinar/realizar inspección (si aplica)",
                "start": "I1_requiere_inspeccion",
                "flags_done": ["S2A_OK"],
            },
            {
                "label": "Rama B (a distancia): Registrar/validar en sistema (según flujo)",
                "start": "I2_notificar_determinacion",
                "flags_done": ["S2B_OK"],
            },
        ],
        "next": "S2_END_UBICACION",
    },

    # Retorno oculto al punto paralelo de Etapa 2
    "S2_RETURN": {
        "type": "parallel_gate",
        "titulo": "Volver a Punto paralelo (sincronización)",
        "rol": "",
        "descripcion": "Regresa al punto paralelo para completar la(s) rama(s) pendiente(s).",
        "ramas": [],
        "next": "S2_G1_paralelo_inicio",
        "hidden": True
    },

    # -----------------------------
    # RAMA A (ON-SITE): Determinar y ejecutar inspección (si aplica)
    # -----------------------------
    "I1_requiere_inspeccion": {
        "type": "decision",
        "titulo": "¿Material requiere inspección técnica?",
        "rol": "Supervisor O&M Solar (on-site)",
        "pregunta": "¿El material recibido requiere inspección técnica?",
        "opciones": [
            {"label": "NO → No requiere inspección técnica (pasar a ubicación)", "next": "I1A_no_requiere_inspeccion"},
            {"label": "SÍ → Requiere inspección técnica", "next": "I7_realizar_inspeccion"},
        ],
        "set_flags_by_choice": {
            "NO → No requiere inspección técnica (pasar a ubicación)": {"S2_REQUIERE_INSP": False},
            "SÍ → Requiere inspección técnica": {"S2_REQUIERE_INSP": True},
        },
        "ayuda": "Corresponde al rombo '¿Material requiere inspección técnica?' del diagrama. Si NO requiere, la rama A se cierra; si SÍ, se ejecuta inspección y se evalúa aprobación.",
    },

    "I1A_no_requiere_inspeccion": {
        "type": "task",
        "titulo": "Registrar determinación: NO requiere inspección técnica",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Registrar/confirmar que el material NO requiere inspección técnica según la determinación realizada.",
        "acciones": [
            "Confirmar que no se ejecutará inspección técnica para este material.",
            "Asegurar trazabilidad de la determinación (criterio/antecedente usado).",
        ],
        "validacion": "¿Quedó confirmada la determinación de que NO requiere inspección técnica?",
        "set_flags": {"S2A_OK": True},
        "next": "S2_G1_paralelo_inicio",
    },

    "I7_realizar_inspeccion": {
        "type": "task",
        "titulo": "Realizar inspección",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Ejecutar inspección técnica del material según criterio técnico/aplicable para el ítem recibido.",
        "acciones": [
            "Realizar inspección técnica del material (según criterio/tipo de material).",
            "Registrar hallazgos relevantes para la aprobación/rechazo.",
        ],
        "validacion": "¿La inspección fue realizada y se registraron hallazgos relevantes?",
        "next": "I8_notificar_resultado_inspeccion",
    },

    "I8_notificar_resultado_inspeccion": {
        "type": "task",
        "titulo": "Notificar resultado inspección técnica",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Notificar el resultado de la inspección técnica al Especialista de Almacén (a distancia) para su verificación/registro según flujo.",
        "acciones": [
            "Enviar notificación del resultado de inspección técnica (aprobado/no aprobado) con evidencia/antecedentes.",
        ],
        "validacion": "¿El resultado de inspección técnica fue notificado con evidencia/antecedentes suficientes?",
        "set_flags": {"S2_RESULTADO_INSP_NOTIFICADO": True},
        "next": "D2_aprueba_inspeccion_onsite",
    },

    "D2_aprueba_inspeccion_onsite": {
        "type": "decision",
        "titulo": "¿Material aprueba inspección?",
        "rol": "Supervisor O&M Solar (on-site)",
        "pregunta": "¿El material aprueba la inspección técnica?",
        "opciones": [
            {"label": "SÍ → Material aprobado para ubicación", "next": "I9A_material_aprobado_ubicacion"},
            {"label": "NO → Preparar material para devolución (Etapa 4)", "next": "I10_preparar_devolucion"},
        ],
        "ayuda": "Corresponde al rombo '¿Material aprueba inspección?' del diagrama. Si NO aprueba, se prepara devolución y se deriva a Etapa 4.",
    },

    "I9A_material_aprobado_ubicacion": {
        "type": "task",
        "titulo": "Material aprobado para ubicación",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Confirmar que el material fue aprobado en inspección técnica y queda habilitado para pasar a almacenamiento/ubicación.",
        "acciones": [
            "Confirmar aprobación del material posterior a inspección técnica.",
        ],
        "validacion": "¿Quedó confirmada la aprobación del material para pasar a almacenamiento/ubicación?",
        "set_flags": {"S2A_OK": True, "S2_APRUEBA_INSP": True},
        "next": "S2_G1_paralelo_inicio",
    },

    "I10_preparar_devolucion": {
        "type": "task",
        "titulo": "Preparar material para devolución",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Preparar el material que NO aprobó inspección técnica para su devolución, según el flujo de rechazo/devolución.",
        "acciones": [
            "Aislar/identificar material rechazado (evitar uso/mezcla).",
            "Preparar el material para devolución según indicación del proceso (embalaje/rotulación/segregación).",
        ],
        "validacion": "¿El material quedó preparado y segregado para devolución?",
        "next": "S2_TO_STAGE4",
    },

    "S2_TO_STAGE4": {
        "type": "stage_end",
        "titulo": "Derivación a Etapa 4 – Rechazo/Devolución",
        "rol": "Sistema",
        "mensaje_fin": "Etapa 2 terminó con NO aprobación de inspección técnica. Continuar con Etapa 4 (Rechazo/Devolución).",
        "next_stage": "S4_START",
        "set_flags": {"S2_DERIVA_S4": True},
    },

    # -----------------------------
    # RAMA B (A DISTANCIA): Notificación + registro/validación en sistema
    # -----------------------------
    "I2_notificar_determinacion": {
        "type": "task",
        "titulo": "Notificar determinación de inspección técnica",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Notificar a Almacén (a distancia) la determinación de inspección técnica para que se gestione el flujo en sistema según corresponda.",
        "acciones": [
            "Notificar determinación de inspección técnica (requiere/no requiere) a Almacén (a distancia).",
        ],
        "validacion": "¿La determinación de inspección técnica fue notificada a Almacén (a distancia)?",
        "next": "I3_evento",
    },

    "I3_evento": {
        "type": "decision",
        "titulo": "¿Evento?",
        "rol": "Especialista de Almacén (a distancia) / Supervisor (confirmación)",
        "pregunta": "¿Existe un evento que habilite avanzar sin esperar plazo (según flujo)?",
        "opciones": [
            {"label": "SÍ → Avanzar (sin esperar 48 hrs)", "next": "I5_registrar_bloqueado_libre"},
            {"label": "NO → Esperar plazo 48 hrs y avanzar", "next": "I4_plazo_48hrs"},
        ],
        "ayuda": "El diagrama incluye un gateway de evento y un temporizador 'Plazo 48 hrs'. Aquí se modela como confirmación operacional del evento o del cumplimiento del plazo.",
    },

    "I4_plazo_48hrs": {
        "type": "task",
        "titulo": "Plazo 48 hrs",
        "rol": "Especialista de Almacén (a distancia) / Supervisor (confirmación)",
        "descripcion": "Confirmar que se cumplió el plazo de 48 horas indicado en el flujo, o que corresponde avanzar según el control temporal definido.",
        "acciones": [
            "Verificar cumplimiento del plazo de 48 horas según flujo.",
        ],
        "validacion": "¿Se cumplió el plazo de 48 horas (o corresponde avanzar según control temporal definido)?",
        "next": "I5_registrar_bloqueado_libre",
    },

    "I5_registrar_bloqueado_libre": {
        "type": "task",
        "titulo": "Registrar material bloqueado en libre utilización",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Registrar/gestionar en sistema el material en condición bloqueada en libre utilización según el flujo de inspección.",
        "acciones": [
            "Registrar el material bloqueado en libre utilización en el sistema según procedimiento.",
        ],
        "validacion": "¿El material quedó registrado en el sistema como bloqueado en libre utilización?",
        "set_flags": {"S2_REG_BLOQ_LIBRE_OK": True},
        "next": "I6_material_a_inspeccion",
    },

    "I6_material_a_inspeccion": {
        "type": "decision",
        "titulo": "¿Material a inspección técnica?",
        "rol": "Especialista de Almacén (a distancia)",
        "pregunta": "Según determinación, ¿el material va a inspección técnica?",
        "opciones": [
            {"label": "NO → Material para almacenamiento/ubicación", "next": "I6A_no_va_a_inspeccion"},
            {"label": "SÍ → Material a inspección (esperar resultado)", "next": "I6B_esperar_resultado"},
        ],
        "ayuda": "Corresponde al rombo '¿Material a inspección técnica?' del diagrama (ramifica a inspección o a almacenamiento).",
    },

    "I6A_no_va_a_inspeccion": {
        "type": "task",
        "titulo": "Material para almacenamiento",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Confirmar que el material NO va a inspección técnica y queda disponible para continuar a almacenamiento/ubicación.",
        "acciones": [
            "Confirmar condición de 'material para almacenamiento' según determinación.",
        ],
        "validacion": "¿Quedó confirmada la condición de material para almacenamiento/ubicación?",
        "set_flags": {"S2B_OK": True},
        "next": "S2_G1_paralelo_inicio",
    },

    "I6B_esperar_resultado": {
        "type": "task",
        "titulo": "Esperar resultado de inspección",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Esperar/recibir la notificación del resultado de inspección técnica desde el Supervisor (on-site) para verificarlo en sistema.",
        "acciones": [
            "Recibir notificación del resultado de inspección técnica (aprobado/no aprobado) con evidencia.",
        ],
        "validacion": "¿Se recibió la notificación del resultado de inspección técnica con evidencia?",
        "next": "I9_verificar_resultado_inspeccion",
    },

    "I9_verificar_resultado_inspeccion": {
        "type": "task",
        "titulo": "Verificar resultado inspección",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Verificar el resultado de la inspección técnica recibido y aplicar el flujo correspondiente (ubicación o devolución).",
        "acciones": [
            "Verificar resultado de inspección técnica recibido.",
        ],
        "validacion": "¿El resultado de inspección fue verificado en sistema con antecedentes suficientes?",
        "next": "D2_aprueba_inspeccion_remoto",
    },

    "D2_aprueba_inspeccion_remoto": {
        "type": "decision",
        "titulo": "¿Material aprueba inspección?",
        "rol": "Especialista de Almacén (a distancia)",
        "pregunta": "Tras verificar resultado, ¿el material aprueba inspección?",
        "opciones": [
            {"label": "SÍ → Material para ubicación en sistema", "next": "I9C_material_para_ubicacion_sistema"},
            {"label": "NO → Gestionar devolución al proveedor (Etapa 4)", "next": "I11_derivar_devolucion"},
        ],
        "ayuda": "Corresponde al rombo '¿Material aprueba inspección?' del diagrama en la pista de Almacén a distancia.",
    },

    "I9C_material_para_ubicacion_sistema": {
        "type": "task",
        "titulo": "Material para ubicación en sistema",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Confirmar que el material aprobado queda habilitado para ubicación/almacenamiento en el sistema.",
        "acciones": [
            "Confirmar condición de material para ubicación en sistema.",
        ],
        "validacion": "¿El material quedó habilitado para ubicación en sistema?",
        "set_flags": {"S2B_OK": True},
        "next": "S2_G1_paralelo_inicio",
    },

    "I11_derivar_devolucion": {
        "type": "task",
        "titulo": "Gestionar devolución al proveedor",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Gestionar la devolución del material rechazado al proveedor y notificar según corresponda (flujo PRO183).",
        "acciones": [
            "Gestionar devolución al proveedor según flujo vigente.",
            "Notificar devolución según corresponda.",
        ],
        "validacion": "¿Se inició/gestionó la devolución al proveedor según el flujo definido?",
        "next": "S2_TO_STAGE4B",
    },

    "S2_TO_STAGE4B": {
        "type": "stage_end",
        "titulo": "Derivación a Etapa 4 – Rechazo/Devolución",
        "rol": "Sistema",
        "mensaje_fin": "Etapa 2 terminó con rechazo por resultado de inspección. Continuar con Etapa 4 (Rechazo/Devolución / PRO183).",
        "next_stage": "S4_START",
        "set_flags": {"S2_DERIVA_S4": True},
    },

    "S2_END_UBICACION": {
        "type": "stage_end",
        "titulo": "Fin Etapa 2 – Inspección",
        "rol": "Sistema",
        "mensaje_fin": "Etapa 2 (Inspección) terminada: material habilitado para ubicación/almacenamiento (con o sin inspección).",
        "next_stage": "S3_START",
        "set_flags": {"S2_OK": True},
    },



    # =============================
# ETAPA 3 — ALMACENAMIENTO / UBICACIÓN
    # (Basado en diagrama PRO200: ubicación física + registro en sistema + etiquetado + notificación disponibilidad si aplica)
    # =============================
    "S3_START": {
        "type": "stage",
        "etapa_num": 3,
        "etapa_nombre": "ALMACENAMIENTO",
        "mensaje": "Etapa 3 (Almacenamiento): desde 'Material para ubicación física / en sistema'. Se determina ubicación, se informa a almacén, se registra ubicación en sistema, se imprime etiqueta y se ubica físicamente el material. Luego se verifica origen de la solicitud y se notifica disponibilidad si aplica.",
        "next": "A0_material_para_ubicacion_fisica",
    },


    "A0_material_para_ubicacion_fisica": {
        "type": "task",
        "titulo": "Material para ubicación física",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Inicio de la Etapa 3 (Almacenamiento). Este hito indica que el material está habilitado para iniciar su ubicación física en almacén.",
        "acciones": [
            "Confirmar que el material quedó habilitado para iniciar ubicación física (según cierre de Etapa 2)."
        ],
        "validacion": "¿El material está habilitado y disponible para iniciar la ubicación física en almacén?",
        "next": "A1_determinar_ubicacion_almacen"
    },

"A1_determinar_ubicacion_almacen": {
        "type": "task",
        "titulo": "Determinar ubicación en almacén",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Determinar la ubicación del material recibido en el almacén, según el subproceso de Almacenamiento del PRO200.",
        "acciones": [
            "Determinar ubicación en almacén para el material recibido (sector/posición definida).",
            "Verificar compatibilidad con requisitos del material (peligrosidad, dimensiones, segregación si aplica)."
        ],
        "validacion": "¿La ubicación en almacén fue determinada (sector/posición) considerando requisitos del material?",
        "next": "A2_informar_ubicacion_almacen",
    },

    "A2_informar_ubicacion_almacen": {
        "type": "task",
        "titulo": "Informar ubicación en almacén",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Informar la ubicación determinada al Especialista de Almacén (a distancia) para su registro en sistema.",
        "acciones": [
            "Informar la ubicación determinada (sector/posición) al Especialista de Almacén (a distancia).",
            "Indicar identificación del material y referencia a OC/recepción."
        ],
        "validacion": "¿La ubicación fue informada al Especialista de Almacén con identificación del material y referencia a OC/recepción?",
        "next": "A0_material_para_ubicacion_sistema",
    },


    "A0_material_para_ubicacion_sistema": {
        "type": "task",
        "titulo": "Material para ubicación en el sistema",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Este hito inicia el tramo de registro en sistema posterior a la ubicación física. Se ejecuta una vez informada correctamente la ubicación física.",
        "acciones": [
            "Confirmar recepción de la notificación de ubicación física enviada por O&M (sector/posición) para proceder al registro en sistema."
        ],
        "validacion": "¿Se recibió correctamente la notificación de ubicación física (sector/posición) para registrar ubicación en el sistema?",
        "next": "A3_registrar_ubicacion_en_sistema"
    },

"A3_registrar_ubicacion_en_sistema": {
        "type": "task",
        "titulo": "Registrar ubicación en sistema",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Registrar la ubicación del material en el sistema, según el diagrama del PRO200.",
        "acciones": [
            "Registrar ubicación del material en el sistema (según transacción/proceso definido para el sitio).",
            "Asegurar que la ubicación registrada coincide con lo informado por Supervisor O&M."
        ],
        "validacion": "¿La ubicación quedó registrada en sistema y coincide con lo informado (sector/posición)?",
        "set_flags": {"S3_UBICACION_REGISTRADA": True},
        "next": "A4_notificar_registro_ubicacion",
    },

    "A4_notificar_registro_ubicacion": {
        "type": "task",
        "titulo": "Notificar registro ubicación en sistema",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Notificar al Supervisor O&M el registro de la ubicación en sistema (material ubicado en sistema).",
        "acciones": [
            "Notificar registro de ubicación en sistema al Supervisor O&M.",
            "Adjuntar/indicar identificador de ubicación registrada (si aplica)."
        ],
        "validacion": "¿Se notificó al Supervisor O&M que el material quedó ubicado en sistema?",
        "next": "A5_imprimir_etiqueta",
    },

    "A5_imprimir_etiqueta": {
        "type": "task",
        "titulo": "Imprimir etiqueta",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Imprimir etiqueta del material para su identificación en almacén, según subproceso Almacenamiento.",
        "acciones": [
            "Imprimir etiqueta del material según estándar del sitio (p.ej., ZMM_IMP_ETIQUETA si aplica).",
            "Verificar que datos de etiqueta correspondan al material (código/descripcion/lote si aplica)."
        ],
        "validacion": "¿La etiqueta fue impresa y verificada (datos corresponden al material)?",
        "next": "A6_ubicar_material_en_almacen",
    },

    "A6_ubicar_material_en_almacen": {
        "type": "task",
        "titulo": "Ubicar material en almacén",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Ubicar físicamente el material en la posición definida en almacén, aplicando etiquetado.",
        "acciones": [
            "Trasladar y ubicar físicamente el material en la ubicación definida (sector/posición).",
            "Aplicar etiqueta y verificar legibilidad/adhesión.",
            "Asegurar condiciones de almacenamiento (seguridad/segregación/orden)."
        ],
        "validacion": "¿El material quedó ubicado físicamente en la ubicación definida y etiquetado (legible)?",
        "next": "A7_notificar_fin_almacenamiento",
    },

    "A7_notificar_fin_almacenamiento": {
        "type": "task",
        "titulo": "Notificar fin del almacenamiento",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Notificar al Especialista de Almacén (a distancia) que el almacenamiento/ubicación física finalizó, según el diagrama.",
        "acciones": [
            "Notificar fin del almacenamiento (material almacenado) al Especialista de Almacén.",
            "Indicar confirmación de ubicación física y etiqueta aplicada."
        ],
        "validacion": "¿Se notificó al Especialista de Almacén el fin del almacenamiento con confirmación de ubicación física?",
        "next": "A8_verificar_origen_solicitud",
    },

    "A8_verificar_origen_solicitud": {
        "type": "task",
        "titulo": "Verificar origen de la solicitud de material",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Verificar el origen de la solicitud de material según el subproceso de almacenamiento del PRO200.",
        "acciones": [
            "Verificar origen de la solicitud de material en sistema (para definir si corresponde notificar disponibilidad)."
        ],
        "validacion": "¿El origen de la solicitud fue verificado para definir necesidad de notificación de disponibilidad?",
        "next": "D3_requiere_informar_disponibilidad",
    },

    "D3_requiere_informar_disponibilidad": {
        "type": "decision",
        "titulo": "¿Requiere informar disponibilidad de material?",
        "rol": "Especialista de Almacén (a distancia)",
        "pregunta": "¿Se requiere informar disponibilidad del material según origen de la solicitud?",
        "opciones": [
            {"label": "NO → Fin recepción", "next": "S3_END"},
            {"label": "SÍ → Notificar disponibilidad de material", "next": "A9_notificar_disponibilidad"},
        ],
        "ayuda": "Este rombo corresponde al diagrama del subproceso: si no se requiere informar, se cierra recepción; si se requiere, se notifica disponibilidad.",
    },

    "A9_notificar_disponibilidad": {
        "type": "task",
        "titulo": "Notificar disponibilidad de material",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Notificar disponibilidad del material cuando corresponde según el flujo de Almacenamiento.",
        "acciones": [
            "Notificar disponibilidad del material a quien corresponda según origen de la solicitud (ej.: solicitante/área).",
            "Incluir ubicación registrada y referencia a recepción."
        ],
        "validacion": "¿La disponibilidad del material fue notificada con ubicación y referencia a recepción?",
        "set_flags": {"S3_DISPONIBILIDAD_NOTIFICADA": True},
        "next": "S3_END",
    },

    "S3_END": {
        "type": "end",
        "titulo": "Fin Etapa 3 – Almacenamiento",
        "rol": "Sistema",
        "mensaje": "Etapa 3 (Almacenamiento) terminada: material ubicado físicamente y registrado/notificado según correspondía. Proceso finalizado.",
        "estado_final": FINALIZADO,
    },


# ETAPA 4 — RECHAZO / DEVOLUCIÓN (PRO183)
    # =============================
    "S4_START": {
        "type": "stage",
        "etapa_num": 4,
        "etapa_nombre": "RECHAZO / DEVOLUCIÓN",
        "mensaje": "Esta etapa se activa cuando el material es rechazado en revisión o no aprueba inspección técnica. Incluye preparación de devolución, registro en sistema y derivación a PRO183 (Devolución a proveedores).",
        "next": "X1_notificar_rechazo_material",
    },

    "X1_notificar_rechazo_material": {
        "type": "task",
        "titulo": "Notificar rechazo de material",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Notificar el rechazo del material recibido.",
        "acciones": [
            "Notificar rechazo de material indicando OC/proveedor y motivo del rechazo.",
            "Adjuntar/indicar guía de despacho u otro documento disponible.",
        ],
        "validacion": "¿Se notificó el rechazo con OC/proveedor/motivo y documento asociado (guía/otros)?",
        "next": "X2_preparar_material_devolucion",
    },

    "X2_preparar_material_devolucion": {
        "type": "task",
        "titulo": "Preparar material para devolución",
        "rol": "Supervisor O&M Solar (on-site)",
        "descripcion": "Asegurar que el material rechazado queda preparado y segregado para devolución al proveedor.",
        "acciones": [
            "Segregar material rechazado en zona definida y segura.",
            "Preparar embalaje/condición para devolución según práctica del sitio y proveedor.",
        ],
        "validacion": "¿El material quedó segregado y preparado para devolución?",
        "next": "X3_verificar_notificacion_recepcion_rechazada",
    },

    "X3_verificar_notificacion_recepcion_rechazada": {
        "type": "task",
        "titulo": "Verificar notificación de recepción rechazada",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Verificar la notificación de recepción rechazada y preparar registro de devolución en sistema.",
        "acciones": [
            "Revisar notificación de rechazo y documentación asociada.",
            "Confirmar datos mínimos para gestionar devolución en SAP.",
        ],
        "validacion": "¿La notificación de rechazo fue verificada y están los datos mínimos para devolución?",
        "next": "X4_registrar_devolucion_libre_utilizacion",
    },

    "X4_registrar_devolucion_libre_utilizacion": {
        "type": "task",
        "titulo": "Registrar devolución en libre utilización",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Registrar devolución en sistema. El PRO menciona movimiento 122 en MIGO para devolución a proveedor.",
        "acciones": [
            "Registrar devolución del material en SAP según corresponda (MIGO -122).",
        ],
        "validacion": "¿La devolución quedó registrada en SAP (MIGO -122) de forma verificable?",
        "next": "X5_generar_devolucion_proveedor",
    },

    "X5_generar_devolucion_proveedor": {
        "type": "task",
        "titulo": "Generar devolución al proveedor",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Generar la devolución al proveedor y asegurar coordinación logística.",
        "acciones": [
            "Gestionar generación de devolución al proveedor según PRO183 (Devolución a proveedores).",
        ],
        "validacion": "¿La devolución al proveedor fue generada según PRO183?",
        "next": "X6_notificar_devolucion_al_proveedor",
    },

    "X6_notificar_devolucion_al_proveedor": {
        "type": "task",
        "titulo": "Notificar devolución al proveedor",
        "rol": "Especialista de Almacén (a distancia)",
        "descripcion": "Notificar que la devolución fue generada y coordinar el retiro/entrega con proveedor y abastecimiento.",
        "acciones": [
            "Notificar devolución al proveedor (y a abastecimiento si corresponde).",
        ],
        "validacion": "¿La devolución fue notificada al proveedor (y abastecimiento si aplica)?",
        "next": "END_PRO183",
    },

    "END_PRO183": {
        "type": "end",
        "titulo": "Derivar a PRO183 – Devolución a proveedores",
        "rol": "",
        "mensaje": "Proceso derivado a PRO183 (Devolución a proveedores) para ejecución logística/administrativa de devolución.",
        "estado_final": DERIVADO_PRO183,
    },
}

# -------------------------
# Motor HMI
# -------------------------

class PRO200HMI:
    def __init__(self):
        self.nodo_id = "S1_START"
        self.historial = []
        self.logs = []
        self.flags = {}  # flags para gates paralelos y estados de etapa
        self.output = widgets.Output(layout={"width":"100%"})

        # Controles estándar
        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        # Controles especiales
        self.btn_continuar = widgets.Button(description="OK – Continuar", button_style="primary", layout={"width":"100%","height":"44px"})
        self.btn_continuar.on_click(self._on_continuar)

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])
        self.block_panel = widgets.VBox([])
        self.is_blocked = False
        self.block_reason = None
        self.btn_rehacer = widgets.Button(description="🔄 Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._decision_widget = None
        self._parallel_buttons = None
        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "nodo_id": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self, prev_id):
        self.historial.append(prev_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    # ---------- UI RENDER ----------

    def _render_header(self, n):
        rol = n.get("rol","")
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;'><b>ROL:</b> {rol}</span>" if rol else ""
        titulo = n.get("titulo","")
        desc = n.get("descripcion","")

        # Etapas: banner diferente
        if n["type"] == "stage":
            etapa_num = n.get("etapa_num")
            etapa_nombre = n.get("etapa_nombre","")
            msg = n.get("mensaje","")
            return widgets.HTML(f"""
            <div style="padding:18px;border-radius:14px;background:#0f172a;border:1px solid #0b1220;color:#ffffff;">
                <div style="font-size:13px;opacity:0.9;"><b>PRO200</b> – Recepción de Materiales a Distancia (Feb-2024)</div>
                <div style="margin-top:10px;font-size:26px;"><b>ETAPA {etapa_num}: {etapa_nombre}</b></div>
                <div style="margin-top:10px;font-size:14px;line-height:1.35;opacity:0.95;">{msg}</div>
            </div>
            """)

        if n["type"] == "stage_end":
            etapa_num = n.get("etapa_num")
            etapa_nombre = n.get("etapa_nombre","")
            msg = n.get("mensaje_fin","")
            return widgets.HTML(f"""
            <div style="padding:18px;border-radius:14px;background:#052e16;border:1px solid #14532d;color:#ecfdf5;">
                <div style="font-size:13px;opacity:0.95;"><b>ETAPA {etapa_num}: {etapa_nombre}</b></div>
                <div style="margin-top:10px;font-size:20px;"><b>✅ Etapa terminada</b></div>
                <div style="margin-top:10px;font-size:14px;line-height:1.35;opacity:0.98;">{msg}</div>
            </div>
            """)

        # Parallel gate: fondo morado para diferenciación
        if n["type"] == "parallel_gate" and not n.get("hidden"):
            return widgets.HTML(f"""
            <div style="padding:16px;border-radius:12px;background:#2e1065;border:1px solid #4c1d95;color:#ffffff;">
                <div style="font-size:12px;opacity:0.9;"><b>PRO200</b> – Motor operacional</div>
                <div style="margin-top:6px;font-size:22px;"><b>🟣 PUNTO PARALELO (AND)</b></div>
                <div style="margin-top:8px;font-size:16px;"><b>{n.get('titulo','')}</b></div>
                <div style="margin-top:10px;font-size:13px;line-height:1.35;opacity:0.95;">{n.get('descripcion','')}</div>
            </div>
            """)

        # Header normal
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO200</b> – Recepción de Materiales a Distancia (Feb-2024)</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{titulo}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;">{desc}</div>
        </div>
        """)

    def _render_task(self, n):
        acciones = "".join([f"<li style='margin:4px 0;color:#0f172a;'>{a}</li>" for a in n.get("acciones",[])])
        valid = n.get("validacion","")
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>⚙️ ACCIÓN A EJECUTAR (texto PRO200)</b></div>
                <ul style="margin-top:10px;padding-left:18px;color:#0f172a;">{acciones}</ul>
            </div>
            """),
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>✅ ¡VALIDACIÓN!</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SÍ</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
            </div>
            """),
        ])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        box = widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo XOR)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ])
        return box, radios

    def _render_parallel_gate(self, n):
        ramas = n.get("ramas",[])
        # estado de flags
        rows=[]
        for r in ramas:
            label=r["label"]
            done=True
            missing=[]
            for f in r.get("flags_done",[]):
                if not self.flags.get(f, False):
                    done=False
                    missing.append(f)
            badge = "✅ COMPLETADO" if done else "⏳ PENDIENTE"
            rows.append((label, badge, missing, r.get("start")))
        # UI
        items=[]
        for (label,badge,missing,start_id) in rows:
            missing_txt = ""
            if missing:
                missing_txt = "<div style='margin-top:6px;font-size:12px;opacity:0.9;'>Faltan: <code>" + ", ".join(missing) + "</code></div>"
            items.append(widgets.HTML(f"""
            <div style="margin-top:10px;padding:12px;border-radius:12px;border:1px solid #6d28d9;background:#ffffff;">
                <div style="font-size:14px;color:#0f172a;"><b>{label}</b></div>
                <div style="margin-top:6px;font-size:13px;color:#0f172a;"><b>Estado:</b> {badge}</div>
                {missing_txt}
            </div>
            """))

        # Botones para ir a ramas pendientes (orden: ramas más cortas primero según definición)
        btns=[]
        for r in ramas:
            label=r["label"]
            start=r["start"]
            # habilitar solo si no completada
            done=True
            for f in r.get("flags_done",[]):
                if not self.flags.get(f, False):
                    done=False
            b=widgets.Button(description=f"Ir a {label}", layout={"width":"100%","height":"40px"})
            b.disabled = done
            def _mk_handler(target):
                def _h(_):
                    self._advance_to(target)
                return _h
            b.on_click(_mk_handler(start))
            btns.append(b)

        can_continue = all(self.flags.get(f, False) for r in ramas for f in r.get("flags_done",[]))
        cont = widgets.Button(description="✅ Todas las ramas completadas – Continuar", button_style="success", layout={"width":"100%","height":"44px"})
        cont.disabled = not can_continue
        def _on_cont(_):
            self._advance_to(n.get("next"))
        cont.on_click(_on_cont)

        return widgets.VBox(items + [widgets.HTML("<div style='height:8px;'></div>")] + btns + [widgets.HTML("<div style='height:8px;'></div>"), cont])

    def _render_stage(self, n):
        # botón iniciar etapa
        b = widgets.Button(description="▶ Iniciar etapa", button_style="primary", layout={"width":"100%","height":"44px"})
        def _go(_):
            self._advance_to(n.get("next"))
        b.on_click(_go)
        return widgets.VBox([widgets.HTML("<div style='height:10px;'></div>"), b])

    def _render_stage_end(self, n):
        self.btn_continuar.description = f"OK – Continuar a Etapa {NODOS[n.get('next_stage')].get('etapa_num')}: {NODOS[n.get('next_stage')].get('etapa_nombre')}"
        self._next_stage_target = n.get("next_stage")
        return widgets.VBox([widgets.HTML("<div style='height:10px;'></div>"), self.btn_continuar])

    def _render_footer(self):
        volver_disabled = (len(self.historial) == 0)
        self.btn_volver.disabled = volver_disabled
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()

            n = NODOS[self.nodo_id]

            # ocultar header/footers en nodos hidden (retornos internos)
            hidden = n.get("hidden", False)

            header = self._render_header(n) if not hidden else widgets.HTML("")

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
            elif n["type"] == "parallel_gate":
                self._decision_widget = None
                body = self._render_parallel_gate(n) if not hidden else widgets.HTML("")
            elif n["type"] == "stage":
                self._decision_widget = None
                body = self._render_stage(n)
            elif n["type"] == "stage_end":
                self._decision_widget = None
                body = self._render_stage_end(n)
            elif n["type"] == "end":
                self._decision_widget = None
                body = widgets.HTML(f"""
                <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
                    <div style="font-size:20px;color:#0f172a;"><b>🏁 FIN / DERIVACIÓN</b></div>
                    <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','')}</div>
                    <div style="margin-top:10px;font-size:12px;color:#0f172a;">Estado final: <b>{n.get('estado_final','')}</b></div>
                </div>
                """)
            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")

            footer = self._render_footer() if not hidden else widgets.HTML("")
            self.main_box.children = [header, body, footer]
            display(self.main_box)

    # ---------- FLOW CONTROL ----------

    def _apply_set_flags(self, n):
        # set_flags directo
        for k,v in (n.get("set_flags") or {}).items():
            self.flags[k] = v
            self._log("FLAG_SET", {"flag": k, "value": v})

    def _apply_set_flags_by_choice(self, n, choice_label):
        mapping = n.get("set_flags_by_choice") or {}
        if choice_label in mapping:
            for k,v in mapping[choice_label].items():
                self.flags[k] = v
                self._log("FLAG_SET", {"flag": k, "value": v})

    def _advance_to(self, next_id):
        prev = self.nodo_id
        if next_id == "G1_RETURN" or next_id == "G2_RETURN":
            # no apilar retorno interno
            self.nodo_id = next_id
            self._log("AVANZA", {"from": prev, "to": next_id})
            # render inmediato para saltar al gate real
            self._render()
            # salto automático al gate real
            self.nodo_id = NODOS[next_id]["next"]
            self._render()
            return

        self._push_hist(prev)
        self.nodo_id = next_id
        self._log("AVANZA", {"from": prev, "to": next_id})
        self._clear_msg()
        self._render()

    def _on_si(self, _):
        if getattr(self, "is_blocked", False):
            self._set_msg("""<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
                <b>⛔ Paso bloqueado:</b> Primero registra el motivo y usa <b>Rehacer paso</b>. Luego valida con <b>SÍ</b>.
            </div>""")
            return

        n = NODOS[self.nodo_id]

        if n["type"] in ("stage","stage_end","parallel_gate","end"):
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> En este nodo usa el botón correspondiente (iniciar/continuar/ramas).</div>")
            return

        if n["type"] == "task":
            self._apply_set_flags(n)
            next_id = n.get("next")
            if next_id:
                self._advance_to(next_id)
            else:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'><b>Atención:</b> Este paso no tiene siguiente definido.</div>")
            return

        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta selección:</b> Elige una opción para avanzar.</div>")
                return

            # set_flags_by_choice si aplica (por label)
            chosen_next = self._decision_widget.value
            # obtener label elegido
            label=None
            for (lab,nxt) in self._decision_widget.options:
                if nxt==chosen_next:
                    label=lab
                    break
            if label:
                self._apply_set_flags_by_choice(n, label)

            self._advance_to(chosen_next)
            return

    def _motivos_bloqueo(self):
        # Genérico para PRO200: se mantiene simple y trazable (sin inventar).
        return [
            "Falta información/documentación para validar (guía/factura/certificados/HDS si aplica)",
            "Discrepancia entre material recibido y OC/SOLPED (código/cantidad/especificación)",
            "Riesgo operacional para descarga/manipulación (requiere control/recursos)",
            "Pendiente validación a distancia (Especialista de Almacén)",
            "Resultado de inspección pendiente / evidencia insuficiente",
        ]

    def _on_no(self, _):
        n = NODOS[self.nodo_id]
        if n["type"] not in ("task","decision"):
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Este nodo no se bloquea con NO.</div>")
            return

        self.is_blocked = True
        self.block_reason = None
        self.btn_si.disabled = True
        self._log("BLOQUEO", {"titulo": n.get("titulo","")})

        motivos = self._motivos_bloqueo()
        radio = widgets.RadioButtons(options=motivos, layout={"width":"100%"}, style={"description_width":"initial"})

        def _on_pick(change):
            if change.get("name") == "value":
                self.block_reason = change["new"]
                self._log("MOTIVO_BLOQUEO_SELECCIONADO", {"motivo": self.block_reason})
                self._set_msg("""
                <div style='margin-top:10px;padding:10px;border-radius:10px;background:#e0f2fe;border:1px solid #0284c7;color:#0f172a;'>
                    <b>✅ Motivo registrado.</b> Ejecuta la corrección y presiona <b>Rehacer paso</b>.
                </div>
                """)

        radio.observe(_on_pick, names="value")

        self.block_panel.children = [
            widgets.HTML("""
            <div style='margin-top:10px;padding:12px;border-radius:12px;background:#fee2e2;border:1px solid #ef4444;color:#0f172a;'>
                <b>🛑 BLOQUEADO:</b> Debes indicar el motivo y luego rehacer el paso.
            </div>
            """),
            widgets.HTML("<div style='margin-top:6px;font-size:13px;color:#0f172a;'><b>Motivo del bloqueo (PRO200):</b></div>"),
            radio,
            widgets.HTML("<div style='height:8px;'></div>"),
            self.btn_rehacer,
        ]

        self._set_msg("""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
            <b>Instrucción:</b> Selecciona un motivo, realiza la corrección/gestión y luego presiona <b>Rehacer paso</b>.
        </div>
        """)

    def _on_rehacer(self, _):
        if not getattr(self, "is_blocked", False):
            self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Este paso no está bloqueado.</div>")
            return
        if not self.block_reason:
            self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta motivo:</b> Debes seleccionar un motivo antes de rehacer.</div>")
            return

        self._log("REHACER_PASO", {"motivo": self.block_reason})
        self.is_blocked = False
        self.btn_si.disabled = False
        self.block_panel.children = []
        self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#e8f5e9;border:1px solid #22c55e;color:#0f172a;'><b>🔄 Paso listo para rehacer.</b> Ejecuta la acción y valida con <b>SÍ</b>.</div>")

    def _on_volver(self, _):
        prev = self._pop_hist()
        self.is_blocked = False
        self.block_reason = None
        self.block_panel.children = []
        self.btn_si.disabled = False
        if prev is None:
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Ya estás en el inicio.</div>")
            return
        cur = self.nodo_id
        self.nodo_id = prev
        self._log("VOLVER", {"from": cur, "to": prev})
        self._clear_msg()
        self._render()

    def _on_continuar(self, _):
        tgt = getattr(self, "_next_stage_target", None)
        if tgt:
            self._advance_to(tgt)

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO200 – Recepción de Materiales a Distancia (Feb-2024)",
            "session_id": str(uuid.uuid4()),
            "export_ts": _now_iso(),
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "flags": dict(self.flags),
            "logs": list(self.logs),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(self.output)

# Iniciar HMI
hmi = PRO200HMI()
hmi.iniciar()

Output(layout=Layout(width='100%'))